# Week 1 — T4 smoke test, baseline, QLoRA SFT

**Kaggle settings:** Accelerator = *GPU T4 x2*, Internet = *on*.
**Secrets** (Add-ons → Secrets, all optional): `HF_TOKEN` (push the merged model), `MLFLOW_TRACKING_URI`, `MLFLOW_TRACKING_USERNAME`, `MLFLOW_TRACKING_PASSWORD`.

All logic lives in the repo; this notebook only calls it. Launched from a laptop with `python scripts/kaggle_run.py run --week 1`, which attaches the exact commit as a private dataset; opened by hand, it falls back to cloning `REPO`. Run top to bottom. The final cell writes `week1_results.zip`. Download it and put its contents in `reports/week1/`.

In [ ]:
REPO = "https://github.com/Rahulverma5/qwen-inference-optimization.git"  # change if your repo differs
HF_REPO = None   # e.g. "<hf-user>/qwen3-1.7b-spider-sft" to push the merged model; needs HF_TOKEN
BASE = "Qwen/Qwen3-1.7B"
OUT = "/kaggle/working/week1"
SPIDER = "/kaggle/working/data/spider_data"

import os
from kaggle_secrets import UserSecretsClient
_s = UserSecretsClient()
for k in ["HF_TOKEN", "MLFLOW_TRACKING_URI", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"]:
    try:
        os.environ[k] = _s.get_secret(k)
    except Exception:
        print(f"secret {k} not set; that integration stays off")
os.makedirs(OUT, exist_ok=True)

In [ ]:
import glob, shutil, subprocess, tarfile
DEST = "/kaggle/working/repo"
marker = glob.glob("/kaggle/input/**/COMMIT", recursive=True)
if marker:  # code uploaded by scripts/kaggle_run.py
    d = os.path.dirname(marker[0])
    os.makedirs(DEST, exist_ok=True)
    if os.path.exists(f"{d}/src.tar"):
        with tarfile.open(f"{d}/src.tar") as t:
            t.extractall(DEST, filter="data")
    else:  # Kaggle unpacked the archive itself
        shutil.copytree(d, DEST, dirs_exist_ok=True)
    commit = open(marker[0]).read().strip()
else:
    !git clone -q {REPO} {DEST}
    commit = subprocess.getoutput(f"git -C {DEST} rev-parse --short HEAD")
open(f"{OUT}/COMMIT", "w").write(commit + "\n")
print("code commit:", commit)
%cd {DEST}
!pip install -q -e . -r requirements-gpu.txt 2>&1 | tail -3

## 1. Environment record (commit this with the results)

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,compute_cap --format=csv | tee {OUT}/gpu.txt
!python -c "import torch; print('torch', torch.__version__, 'cuda', torch.version.cuda, 'bf16', torch.cuda.is_bf16_supported())" | tee {OUT}/torch.txt
!pip freeze | grep -iE "^(vllm|torch|transformers|trl|peft|bitsandbytes|accelerate|datasets|llmcompressor|mlflow)==" | tee {OUT}/versions.txt

## 2. Data (pinned + checksum-verified)

In [ ]:
!python scripts/get_spider.py --out /kaggle/working/data

## 3. vLLM smoke test on T4 (gate)
Expect fp16 (T4 has no bf16). Note which attention backend vLLM picks in the log. If this fails, pin an older vLLM in `requirements-gpu.txt` and record why in `docs/weeks/week1.md`.

In [ ]:
!python -c "from vllm import LLM; llm = LLM('{BASE}', dtype='float16', max_model_len=2048); print(llm.generate(['SELECT'])[0].outputs[0].text)" 2>&1 | tee {OUT}/vllm_smoke.log | grep -iE "backend|error|SELECT|dtype" | head -20

## 4. Baseline: stock Qwen3-1.7B on the full Spider dev set (1,034)

In [ ]:
!python -m evaluation.generate --model {BASE} --spider-dir {SPIDER} --out {OUT}/preds/base_fp16.jsonl --max-model-len 2048

## 5. SFT smoke run (64 examples) — catches OOM / template problems in minutes

In [ ]:
!python -m finetune.train_sft --spider-dir {SPIDER} --limit 64 --epochs 1 --out /kaggle/working/sft_smoke --report-to none

## 6. Full QLoRA SFT on both T4s (DDP)
Checkpoints every 200 steps. If the session dies, re-run this cell: training resumes from the last checkpoint.

In [ ]:
!accelerate launch --num_processes 2 --mixed_precision fp16 -m finetune.train_sft --spider-dir {SPIDER} --out /kaggle/working/sft --run-name r32-lr2e-4-e2 2>&1 | tee {OUT}/train.log | grep -vE "it/s\]$" | tail -40

## 7. Merge LoRA into the fp16 base, evaluate the fine-tuned model

In [ ]:
push = f"--push {HF_REPO}" if HF_REPO else ""
!python -m finetune.merge --base {BASE} --adapter /kaggle/working/sft/adapter --out /kaggle/working/sft-merged {push}
!python -m evaluation.generate --model /kaggle/working/sft-merged --spider-dir {SPIDER} --out {OUT}/preds/sft_fp16.jsonl --max-model-len 2048

## 8. Collect results

In [ ]:
import glob, json, shutil
shutil.copy("/kaggle/working/sft/trainer_state.json", f"{OUT}/trainer_state.json")
shutil.copy("/kaggle/working/sft/adapter/adapter_config.json", f"{OUT}/adapter_config.json")
for f in sorted(glob.glob(f"{OUT}/preds/*.score.json")):
    print(f, json.load(open(f)))
# Keep sft-merged/ and sft/adapter/ as kernel output (Week 3 attaches them); drop bulky scratch.
for d in ["/kaggle/working/data", "/kaggle/working/sft_smoke", *glob.glob("/kaggle/working/sft/checkpoint-*")]:
    shutil.rmtree(d, ignore_errors=True)
shutil.make_archive("/kaggle/working/week1_results", "zip", OUT)
print("download /kaggle/working/week1_results.zip")